## Amazon AgentCore Bedrock Code Interpreter 튜토리얼

이 튜토리얼에서는 AgentCore Bedrock Code Interpreter를 사용하여 다음 작업을 수행하는 방법을 알아봅니다.
1. 샌드박스 환경 설정
2. 데이터 로드 및 분석
3. 샌드박스 환경에서 코드 실행
4. 결과 처리 및 가져오기

## 사전 요구 사항
- Bedrock AgentCore Code Interpreter에 액세스할 수 있는 AWS 계정
- Code Interpreter 리소스를 생성하고 관리하는 데 필요한 IAM 권한
- 필수 Python 패키지 설치(boto3 및 bedrock-agentcore 포함)
- 샘플 데이터 파일(data.csv)
- 분석 스크립트(stats.py)


## IAM 실행 역할에 다음 IAM 정책을 연결해야 합니다

~~~ {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:CreateCodeInterpreter",
                "bedrock-agentcore:StartCodeInterpreterSession",
                "bedrock-agentcore:InvokeCodeInterpreter",
                "bedrock-agentcore:StopCodeInterpreterSession",
                "bedrock-agentcore:DeleteCodeInterpreter",
                "bedrock-agentcore:ListCodeInterpreters",
                "bedrock-agentcore:GetCodeInterpreter"
            ],
            "Resource": "*"
        },
        {
            "Effect": "Allow",
            "Action": [
                "logs:CreateLogGroup",
                "logs:CreateLogStream",
                "logs:PutLogEvents"
            ],
            "Resource": "arn:aws:logs:*:*:log-group:/aws/bedrock-agentcore/code-interpreter*"
        }
    ]
}



## 작동 방식

코드 실행 샌드박스는 Code Interpreter, 셸, 파일 시스템을 갖춘 격리 환경을 생성하여 에이전트가 사용자 질의를 안전하게 처리할 수 있도록 합니다. 대규모 언어 모델(LLM)이 도구 선택을 지원한 후 이 세션 내에서 코드가 실행되며, 결과는 종합을 위해 사용자 또는 에이전트에게 반환됩니다.

![로컬 아키텍처](code-interpreter.png)

## 1. 환경 설정

먼저 필요한 라이브러리를 가져오고 Code Interpreter 클라이언트를 초기화합니다.

In [ ]:
!pip install --upgrade -r requirements.txt

In [21]:
from bedrock_agentcore.tools.code_interpreter_client import CodeInterpreter
import json
from typing import Dict, Any

# AWS 리전에 맞게 Code Interpreter 초기화
code_client = CodeInterpreter("us-west-2")
code_client.start()

'01K00Z3F8WZ9KBBW4QGRJCVBHH'

## 2. 로컬 파일 읽기

이제 샘플 데이터 파일과 분석 스크립트의 내용을 읽습니다.

In [22]:
def read_file(file_path: str) -> str:
    """오류 처리를 포함해 파일 내용을 읽는 헬퍼 함수"""
    try:
        with open(file_path, "r", encoding="utf-8") as file:
            return file.read()
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        return ""
    except Exception as e:
        print(f"An error occurred: {e}")
        return ""


# 두 파일 읽기
data_file_content = read_file("samples/data.csv")
code_file_content = read_file("samples/stats.py")

## 3. 샌드박스 환경용 파일 준비

샌드박스 환경에서 생성할 파일을 정의하는 구조를 만듭니다.

In [23]:
files_to_create = [
    {"path": "data.csv", "text": data_file_content},
    {"path": "stats.py", "text": code_file_content},
]

## 4. 도구 호출용 헬퍼 함수 생성

이 헬퍼 함수를 사용하면 샌드박스 도구를 더 쉽게 호출하고 응답을 처리할 수 있습니다. 활성 세션에서는 지원되는 언어(Python, JavaScript)로 코드를 실행하고, 종속성 구성에 따른 라이브러리에 액세스하고, 시각화를 생성하고, 실행 간 상태를 유지할 수 있습니다.

In [24]:
def call_tool(tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
    """샌드박스 도구를 호출하는 헬퍼 함수

    매개변수:
        tool_name (str): 호출할 도구 이름
        arguments (Dict[str, Any]): 도구에 전달할 인수

    반환값:
        Dict[str, Any]: JSON 형식의 결과
    """
    response = code_client.invoke(tool_name, arguments)
    for event in response["stream"]:
        return json.dumps(event["result"])

## 5. 샌드박스에 파일 쓰기

이제 파일을 샌드박스 환경에 쓰고 정상적으로 생성되었는지 확인합니다.

In [25]:
# 샌드박스에 파일 쓰기
writing_files = call_tool("writeFiles", {"content": files_to_create})
print("Writing files result:")
print(writing_files)

# 파일이 생성되었는지 확인
listing_files = call_tool("listFiles", {"path": ""})
print("\nFiles in sandbox:")
print(listing_files)

Writing files result:
{"content": [{"type": "text", "text": "Successfully wrote all 2 files"}], "isError": false}

Files in sandbox:
{"content": [{"type": "resource_link", "uri": "file:///log", "name": "log", "description": "Directory"}, {"type": "resource_link", "mimeType": "text/csv", "uri": "file:///data.csv", "name": "data.csv", "description": "File"}, {"type": "resource_link", "mimeType": "text/x-python", "uri": "file:///stats.py", "name": "stats.py", "description": "File"}, {"type": "resource_link", "uri": "file:///.ipython", "name": ".ipython", "description": "Directory"}], "isError": false}


## 6. 분석 실행

이제 샌드박스 환경에서 분석 스크립트를 실행하고 결과를 처리합니다.

In [26]:
import pprint

# 분석 스크립트 실행
code_execute_result = call_tool(
    "executeCode",
    {"code": files_to_create[1]["text"], "language": "python", "clearContext": True},
)

# 결과 파싱 및 표시
analysis_results = json.loads(code_execute_result)
print("Full analysis results:")
pprint.pprint(analysis_results)

print("\nStandard output from analysis:")
print(analysis_results["structuredContent"]["stdout"])

Full analysis results:
{'content': [{'text': 'Name   Place  Animal   Thing\n'
                      'count       299130  299130  299130  299130\n'
                      'unique        1722      55      50      51\n'
                      'top     Lisa White  Prague    Goat  Pencil\n'
                      'freq           222    5587    6141    6058',
              'type': 'text'}],
 'isError': False,
 'structuredContent': {'executionTime': 0.7111423015594482,
                       'exitCode': 0,
                       'stderr': '',
                       'stdout': 'Name   Place  Animal   Thing\n'
                                 'count       299130  299130  299130  299130\n'
                                 'unique        1722      55      50      51\n'
                                 'top     Lisa White  Prague    Goat  Pencil\n'
                                 'freq           222    5587    6141    6058'}}

Standard output from analysis:
Name   Place  Animal   Thing
count       29

## 7. 정리

마지막으로 Code Interpreter 세션을 중지하여 정리합니다. 세션 사용이 끝나면 리소스를 해제하고 불필요한 비용이 발생하지 않도록 세션을 중지해야 합니다.

In [27]:
# Code Interpreter 세션 중지
code_client.stop()
print("Code Interpreter session stopped successfully!")

Code Interpreter session stopped successfully!


## 마무리

이 튜토리얼에서는 다음 방법을 알아보았습니다.
- Code Interpreter 세션 초기화
- 분석할 파일 읽기 및 준비
- 샌드박스 환경에서 코드 실행
- 결과 처리 및 표시
- 리소스 정리